# Multi-parcellation workflow: your data in siibra, as a reproducible lab book

In this tutorial you bring **your own NIfTI images** (fMRI, PET, ...) registered to MNI152 and use
[siibra](https://siibra-python.readthedocs.io) to

1. load your images as siibra volumes,
2. extract region-wise signals with **three parcellations** at once,
3. compare what the different parcellations say about your data,
4. compare labelled and probabilistic versions of the same atlas, and
5. keep everything in this notebook as a **lab book** that colleagues can re-run on the same data.

No data at hand? Part 0 downloads preprocessed fMRI runs from a public dataset (AOMIC-PIOP2 on OpenNeuro).

### What your data should look like

* **N ≥ 1** `.nii.gz` files, one per subject/session/condition
* all registered to **MNI152** and with the **same shape**
* **either all 3D** (e.g. statistical maps, PET images) **or all 4D** (e.g. preprocessed BOLD) — not mixed

### Project layout

All paths are **relative to the notebook's folder**. Keep this layout and share the whole folder; a
colleague can then run *Restart & Run All* and get the same results.

```
my_project/
├── multi_parcellation_workflow.ipynb   <- this lab book
├── data/                               <- your .nii.gz files (or the downloaded example data)
└── results/                            <- generated tables
```

## 0. Setup, settings, example data (≈ 10 min)

In [ ]:
import json
import pathlib
import platform
import re
from datetime import datetime

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import requests
from nilearn import image, plotting

import siibra

### Your settings

This is the **only cell you need to edit** to run the workflow on your own data.

In [ ]:
# --- input data (paths relative to this notebook) -----------------------------
DATA_DIR = "data"           # folder containing your .nii.gz files
SUFFIX = "nii.gz"
RECURSIVE = False           # also search subfolders
SPACE = siibra.spaces.get("mni152")

# --- parcellations --------------------------------------------------------------
selected_parcellations = ["julich 3.1", "DiFuMo 64"]

# --- analysis -----------------------------------------------------------------
STRATEGY = "mean"           # how voxels are summarized within a region
TOP_N = 15                  # number of top regions shown per parcellation
SEED = 25                   # random seed for sampling in Part 4 (fixed for reproducibility)

# --- output (paths relative to this notebook) ---------------------------------
RESULTS_DIR = pathlib.Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print(SPACE)

### Example data (only if you did not bring any)

If `DATA_DIR` contains no `.nii.gz` files, this cell downloads the dataset used in siibra's fMRI workflow
example: **AOMIC-PIOP2** ([OpenNeuro ds002790](https://openneuro.org/datasets/ds002790/versions/2.0.0)),
preprocessed with fMRIPrep. For each subject we fetch the working-memory run in `MNI152NLin2009cAsym`
space. The list of subjects comes from the dataset's `participants.tsv`; subjects without a
working-memory run are skipped.

Each file is a full 4D BOLD run, so start with a few subjects. Set `N_SUBJECTS = None` to download all of them.

In [ ]:
DATASET_URL = "https://s3.amazonaws.com/openneuro.org/ds002790/"
FILE_TEMPLATE = (
    "derivatives/fmriprep/{subject}/func/"
    "{subject}_task-workingmemory_acq-seq_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
)
N_SUBJECTS = 3

pathlib.Path(DATA_DIR).mkdir(exist_ok=True)
repo = siibra.retrieval.repositories.LocalFileRepository(DATA_DIR)

if not repo.search_files(suffix=SUFFIX, recursive=RECURSIVE):
    participants = pd.read_csv(DATASET_URL + "participants.tsv", sep="\t")
    for subject in participants["participant_id"][:N_SUBJECTS]:
        url = DATASET_URL + FILE_TEMPLATE.format(subject=subject)
        response = requests.get(url, stream=True)
        if response.status_code != 200:
            print(f"{subject}: no working-memory run - skipped")
            continue
        out = pathlib.Path(DATA_DIR) / url.split("/")[-1]
        partial = out.with_name(out.name + ".part")  # renamed once complete, so interrupted downloads are not picked up
        with open(partial, "wb") as fp:
            for chunk in response.iter_content(chunk_size=2**20):
                fp.write(chunk)
        partial.rename(out)
        print("downloaded", out)
else:
    print(f"Found data in {repo} - nothing downloaded.")

## 1. Bring your data

### 1.1 Read the niftis as siibra volumes

siibra reads data through *repository connectors* — the same abstraction it uses for its own
configuration on GitHub. The `LocalFileRepository` created above wraps your data folder, and
`search_files` returns the matching files **relative to that folder**, which keeps the lab book portable.

Each file becomes a siibra volume with `siibra.volumes.from_file`, which ties it to a reference space so it
can be combined with any map in that space. The image data is only read when needed: fetching a volume
gives a nibabel image whose header (shape, voxel size) is available right away.

In the same step we build an inventory. It lists BIDS-style `key-value` entities found in the file names
(e.g. `sub`, `ses`, `task`, `run`), so you can see at a glance which subjects and conditions are present.
The first image tells us whether we are working with time series (4D) or single volumes (3D).

In [ ]:
volumes = {}
rows = []
for relpath in sorted(repo.search_files(suffix=SUFFIX, recursive=RECURSIVE)):
    path = pathlib.Path(DATA_DIR) / relpath
    fname = path.name
    volumes[fname] = siibra.volumes.from_file(path.as_posix(), space=SPACE, name=fname)
    img = volumes[fname].fetch()  # header only; data stays on disk until used
    rows.append(dict(
        filename=fname,
        **dict(re.findall(r"([A-Za-z0-9]+)-([A-Za-z0-9]+)", fname)),  # BIDS-style entities, if any
        path=path.as_posix(),
        shape=img.shape,
        voxel_mm=tuple(round(float(z), 2) for z in img.header.get_zooms()[:3]),
    ))
inventory = pd.DataFrame(rows).set_index("filename")
filenames = list(volumes)

first_shape = inventory["shape"].iloc[0]
is_timeseries = len(first_shape) == 4 and first_shape[3] > 1
N = len(inventory)
print(f"N = {N} {'4D time series' if is_timeseries else '3D images'} of shape {first_shape}")
inventory

### 1.2 Look at the data

`nilearn.plotting.view_img` opens an interactive viewer — click to navigate, hover to read values.
We look at one image (for 4D data, its temporal mean).

In [ ]:
for fname in filenames:
    img = volumes[fname].fetch()
    img3d = image.mean_img(img) if is_timeseries else img
    plotting.plot_glass_brain(
        img3d,
        title=f"{fname}{' (temporal mean)' if is_timeseries else ''}",
        cmap="magma",
        symmetric_cbar=False,
    )

## 2. Choose maps

### 2.1 What is available?

`siibra.maps` contains all preconfigured parcellation maps. We filter for labelled maps in our space.

In [ ]:
for mp in siibra.maps:
    if mp.space == SPACE and mp.is_labelled:
        print(mp)

### 2.2 Select two lenses

Each parcellation is a different *lens* on the same data:

| Name | Principle |
|---|---|
| **Julich-Brain 3.1** | cytoarchitecture (microstructure), from 10 post-mortem brains |
| **DiFuMo 64** | functional modes learned from large fMRI collections |

We use the **labelled** version of each map: every voxel belongs to exactly one region, and signals are
summarized within regions with `STRATEGY`. The selection is set by `selected_parcellations` in the settings
cell, and `maps` uses the same strings as keys, e.g. `maps["julich 3.1"]`.

In [ ]:
maps = {
    parcellation: siibra.get_map(parcellation, SPACE, "labelled")
    for parcellation in selected_parcellations
}

pd.DataFrame([
    dict(selection=parcellation, name=parcellation_map.parcellation.name, n_regions=len(parcellation_map.regions))
    for parcellation, parcellation_map in maps.items()
]).set_index("selection")

## 3. Region-wise signal extraction and comparison

`Map.extract_signals_with_nilearn(volume)` returns a table with **one column per region** and **one
row per time point** (one row for 3D images). siibra builds the nilearn masker for the map, resamples the
map to your image grid, and names the columns after the regions.

* Small regions can disappear when a fine map is resampled to a coarse image (e.g. 4 mm fMRI); siibra
  reports them and fills them with zeros. All-zero regions are excluded from the rankings below.
* Extracted tables are saved to `results/signals/` so that you, and colleagues, can inspect them.

In [ ]:
SIGNAL_DIR = RESULTS_DIR / "signals"


def extract(parc_map, fname, strategy=STRATEGY):
    df = parc_map.extract_signals_with_nilearn(
        volumes[fname], strategy=strategy, verbose=0
    )
    out = (
        SIGNAL_DIR
        / parc_map.name.replace(" ", "_")
        / f"{fname.removesuffix('.' + SUFFIX)}_{strategy}.csv"
    )
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out)
    return df


signals = {
    parcellation: {fname: extract(mp, fname) for fname in filenames}
    for parcellation, mp in maps.items()
}
for parcellation, sig in signals.items():
    print(
        f"{parcellation:18s}: {N} table(s) of shape {sig[filenames[0]].shape} (samples x regions)"
    )

signals["julich 3.1"][filenames[0]].iloc[:5, :6]

### 3.1 Visualize the signals using the maps

`Map.colorize(values)` paints each region with a value, giving a "parcellated" version of your data.

In [ ]:
fig, axs = plt.subplots(len(maps) * len(SUMMARIES), 1, squeeze=False,
                        figsize=(14, 2.8 * len(maps) * len(SUMMARIES)))
panels = [(sname, parcellation) for sname in SUMMARIES for parcellation in maps]
for ax, (sname, parcellation) in zip(axs[:, 0], panels):
    values = profiles[sname][parcellation].mean()  # all regions, in the order siibra produced them
    signed = (informative(profiles[sname][parcellation]) < 0).any().any()
    plotting.plot_stat_map(
        maps[parcellation].colorize(values), axes=ax, display_mode="z", cut_coords=7,
        title=f"{parcellation} — {sname}", cmap="RdBu_r" if signed else "magma",
        symmetric_cbar=signed, annotate=False,
    )

### HANDS ON: 3.2 From signals to regional profiles

To compare subjects we reduce each table to one value per region — a **regional profile**:

* 3D input: the single row (`value`);
* 4D input: two profiles, the temporal **mean** (average signal level) and the temporal **std**
  (fluctuation amplitude, as in siibra's fMRI workflow example).

Stacking the profiles gives a *subjects × regions* table per parcellation and summary.

### HANDS ON: 3.3 Top regions per parcellation

Bars show the group mean, error bars the standard deviation across subjects (one row per summary for
4D data). Each parcellation has its own colour from matplotlib's paired `tab20` colormap; the darker
shade is used for the temporal mean (or the 3D value), the lighter shade for the temporal std. Read the panels side by side: do cytoarchitectonic areas, gyral labels, and functional modes
tell the same story?

### HANDS ON: 3.4 Variability across subjects

Each subject's profile for the top regions, z-scored per region across subjects so that the pattern of
each row is visible regardless of its scale.

In [ ]:
subject_ids = "sub-" + inventory["sub"]  # file name -> subject id, from the inventory



## 4. Cross-parcellation view: clusters and region assignment

Region-wise extraction answers *"how much signal is in each region?"*. The complementary question is
*"where are the main structures in my data, and what does each parcellation call them?"*

1. Build one **group summary image**: the mean across subjects (of the temporal mean for 4D data).
2. **Draw random points** from it, treating its values as an unnormalized probability distribution.
   `e` sharpens the distribution (higher = only the strongest voxels), `sigma_mm` attaches an uncertainty.
3. **Cluster** the points with HDBSCAN and build a kernel-density **cluster map** for each cluster.
4. **Assign** each cluster map to every parcellation; the assignment reports correlation and overlap
   between the cluster and each region.

In [ ]:
subject_imgs = [image.mean_img(vol.fetch()) if is_timeseries else vol.fetch() for vol in volumes.values()]
group_img = image.mean_img(subject_imgs)

# wrap the nibabel image as a siibra volume in our space
group_volume = siibra.volumes.from_nifti(group_img, space=SPACE, name="group summary image")
plotting.plot_glass_brain(group_img, plot_abs=False, colorbar=True, title=f"Group summary image (N = {N})")

In [ ]:
N_SAMPLES = 1000
SHARPNESS = 5        # exponent e: higher -> only the strongest voxels are sampled
SIGMA_MM = 3         # spatial uncertainty attached to each sample point
INVERT = False       # True: sample the lowest values instead (e.g. deactivations in a t-map)

np.random.seed(SEED)
samples = group_volume.draw_samples(N_SAMPLES, e=SHARPNESS, sigma_mm=SIGMA_MM, invert=INVERT)
samples.find_clusters(min_fraction=1 / 300, max_fraction=1 / 2)

points = np.array(samples.as_list())
cluster_info = pd.DataFrame([
    dict(cluster=cl, n_points=int((samples.labels == cl).sum()),
         centroid_mm=tuple(np.round(points[samples.labels == cl].mean(axis=0), 1)))
    for cl in sorted(set(samples.labels) - {-1})
]).set_index("cluster").sort_values("n_points", ascending=False)
print(f"{len(cluster_info)} clusters, {(samples.labels == -1).sum()} unclustered points")

view = plotting.plot_glass_brain(group_img, plot_abs=False, alpha=0.6, title="Sampled points by cluster")
mask = samples.labels >= 0
view.add_markers(points[mask], marker_size=5,
                 marker_color=[samples.label_colors[cl] for cl in samples.labels[mask]])
cluster_info

Now build a cluster map for each cluster and assign it to every parcellation. For each pair we keep
the best-correlated region. This table is the core of the cross-parcellation comparison.

In [ ]:
MIN_CORRELATION = 0.1

cluster_vols = {
    cl: siibra.volumes.from_pointcloud(samples, label=cl, target=group_volume)
    for cl in cluster_info.index
}

assignments = {}  # (cluster, parcellation) -> assignment table
rows = []
for cl, cvol in cluster_vols.items():
    row = {"cluster": cl, **cluster_info.loc[cl].to_dict()}
    for parcellation, parcellation_map in maps.items():
        with siibra.QUIET:
            a = parcellation_map.assign(cvol, split_components=False)
        a = a[a["correlation"] >= MIN_CORRELATION].sort_values("correlation", ascending=False)
        assignments[cl, parcellation] = a
        row[parcellation] = f"{a['region'].iloc[0].name} (r={a['correlation'].iloc[0]:.2f})" if len(a) else "-"
    rows.append(row)

cross_table = pd.DataFrame(rows).set_index("cluster")
cross_table.to_csv(RESULTS_DIR / "cross_parcellation_assignment.csv")
cross_table

### 4.1 Zoom into one cluster

By default we look at the largest cluster; change `SELECTED_CLUSTER` to explore others. The plots
overlay the cluster (red) on the best-matching region of each parcellation (blue).

In [ ]:
SELECTED_CLUSTER = cluster_info.index[0]
cluster_img = cluster_vols[SELECTED_CLUSTER].fetch()

cols = ["region", "correlation", "intersection over union", "map containedness", "input containedness"]
for parcellation in maps:
    print(f"\n{parcellation}")
    display(assignments[SELECTED_CLUSTER, parcellation][cols].head(5))

best_regions = {
    parcellation: assignments[SELECTED_CLUSTER, parcellation]["region"].iloc[0]
    for parcellation in maps if len(assignments[SELECTED_CLUSTER, parcellation])
}
fig, axs = plt.subplots(len(best_regions), 1, figsize=(10, 3.2 * len(best_regions)), squeeze=False)
for ax, (parcellation, region) in zip(axs[:, 0], best_regions.items()):
    view = plotting.plot_glass_brain(region.get_regional_mask(SPACE).fetch(), axes=ax, cmap="Blues",
                                     title=f"{parcellation}: {region.name}")
    view.add_overlay(cluster_img, cmap="Reds", alpha=0.6)

## 5. Labelled vs. probabilistic Julich-Brain (≈ 10 min)

The labelled Julich-Brain map is derived from **probability maps**: each area has its own map giving,
per voxel, the probability of finding that area there. The labelled map keeps only the most probable
area per voxel. With the probabilistic (statistical) map, nilearn projects the data onto all
overlapping maps at once instead of averaging within hard borders.

Julich-Brain 3.1 has two statistical maps in MNI152 (414 and 454 regions); `spec="207"` selects the one
whose 414 regions match the labelled map. We extract signals for one image and compare the regional
profiles. Note that the statistical map is a large download and the projection is memory-intensive.

In [ ]:
julich_pmaps = siibra.get_map("julich 3.1", SPACE, "statistical", spec="207")
print(julich_pmaps.name, "-", len(julich_pmaps.regions), "probability maps")

fname = filenames[0]
summarize = SUMMARIES[MAIN_SUMMARY]
p_labelled = informative(profiles[MAIN_SUMMARY]["julich 3.1"].loc[[fname]]).iloc[0]
p_stat = summarize(julich_pmaps.extract_signals_with_nilearn(volumes[fname], verbose=0))

common = p_labelled.index.intersection(p_stat.index)
rho = p_labelled[common].corr(p_stat[common], method="spearman")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(p_labelled[common], p_stat[common], s=8)
ax.set(xlabel=f"labelled ({STRATEGY})", ylabel="probabilistic (projection)",
       title=f"Julich-Brain 3.1, {fname}: Spearman rho = {rho:.2f} ({len(common)} regions)")

pd.DataFrame({
    "labelled": p_labelled[common].sort_values(ascending=False).index[:10],
    "probabilistic": p_stat[common].sort_values(ascending=False).index[:10],
}, index=pd.RangeIndex(1, 11, name="rank"))

## 6. Lab book: provenance and sharing

* **Same data:** the inventory from Part 1 is written to `results/` so a colleague can see which files were used.
* **Same software:** the environment cell records the versions used.
* **Same randomness:** sampling in Part 4 uses the fixed `SEED`.
* **Same layout:** all paths are relative to the notebook folder.

To share, send the whole project folder (notebook, `data/`, `results/`). Your colleague installs the
listed versions, opens the notebook, and runs *Kernel → Restart & Run All*.

In [ ]:
provenance = {
    "run_at": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        "siibra": siibra.__version__,
        "nilearn": __import__("nilearn").__version__,
        "nibabel": nib.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": __import__("sklearn").__version__,
    },
    "settings": dict(
        DATA_DIR=DATA_DIR, SPACE=SPACE.name, STRATEGY=STRATEGY, TOP_N=TOP_N, SEED=SEED,
        selected_parcellations=selected_parcellations,
    ),
}
(RESULTS_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2))
inventory.to_csv(RESULTS_DIR / "data_inventory.csv")
print(json.dumps(provenance, indent=2))

### Your lab notes

* Date / author:
* Data:
* Observations:
* Open questions / next steps:

#### a few sample questions

* Which regions rank highest in each parcellation, and do the three lenses agree?
* For 4D data, do the mean and std profiles highlight the same regions?
* Which cluster in the cross-parcellation table is most consistently named?
* How much do the labelled and probabilistic Julich-Brain profiles differ?

## Wrap-up

You turned a folder of NIfTI files into

* siibra volumes in a reference space,
* region-wise signals for **threetwo parcellations** with different organizing principles,
* a **cross-parcellation table** naming each cluster in cytoarchitectonic and functional terms,
* a comparison of **labelled and probabilistic** Julich-Brain profiles, and
* a **reproducible lab book** that colleagues can re-run on the same data.

### Where to go from here

* **Clean signals:** `extract_signals_with_nilearn` accepts `confounds=` and `sample_mask=`.
* **Share data with siibra configurations:** describe your images in JSON and load them with
  `siibra.extend_configuration` (see siibra's fMRI analysis workflow example).
* **Docs:** [siibra-python documentation](https://siibra-python.readthedocs.io).